# Mulit Step DQN

In [ ]:
!git clone https://github.com/icu-sepsis/icu-sepsis.git
%cd icu-sepsis/packages/
!pip install icu_sepsis/

In [ ]:
import gymnasium as gym
import icu_sepsis

import numpy as np
import matplotlib
matplotlib.rcParams['agg.path.chunksize'] = 10000
import matplotlib.pyplot as plt
import os
import re
import glob
import random
from itertools import product
from pathlib import Path

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

from gymnasium.vector import SyncVectorEnv

from collections import deque

from typing import List

env = gym.make('Sepsis/ICU-Sepsis-v2')

state, info = env.reset()
print('Initial state:', state)
print('Extra info:', info)

next_state, reward, terminated, truncated, info = env.step(0)
print('\nTaking action 0:')
print('Next state:', next_state)
print('Reward:', reward)
print('Terminated:', terminated)
print('Truncated:', truncated)

print("Actions and State Space:")

print("Action space:", env.action_space)
print("State space:", env.observation_space)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

def one_hot(idx: torch.LongTensor, size: int, device=None):
    return F.one_hot(idx, num_classes=size).to(torch.float32).to(device)

# n-step FIFO buffer (one per env)
class NStepBuffer:
    """Collect up to n transitions and emit an n-step return."""
    def __init__(self, n: int, gamma: float):
        self.n, self.g = n, gamma
        self.buf = deque()                # holds (s,a,r,d)

    def push(self, s, a, r, d):
        self.buf.append((s, a, r, d))

    def _build_transition(self):
        R, gp = 0.0, 1.0                  # return & γ^k multiplier
        for _, _, r, d in self.buf:
            R  += gp * r
            gp *= self.g
            if d: break
        s0, a0, _, _ = self.buf[0]
        sN, _, _, dN = self.buf[min(self.n-1, len(self.buf)-1)]
        return s0, a0, R, sN, dN, gp

    def step(self, s, a, r, d):
        """Returns list with ≥0 ready n-step transitions."""
        ready = []
        self.push(s, a, r, d)
        if len(self.buf) >= self.n or d:      # enough steps OR episode ended
            ready.append(self._build_transition())
            self.buf.popleft()
        if d:                                 # clear on episode end
            self.buf.clear()
        return ready

# Prioritised replay (contiguous arrays, stores γ^k factor)
class PrioritizedReplayNStep:
    def __init__(self, capacity: int, alpha: float=0.6):
        self.cap, self.alpha = capacity, alpha
        self.pos  = 0
        self.full = False
        # arrays --------------------------------------------------
        self.s  = np.empty(capacity, dtype=np.int32)
        self.a  = np.empty(capacity, dtype=np.int16)
        self.R  = np.empty(capacity, dtype=np.float32)   # n-step return
        self.s2 = np.empty(capacity, dtype=np.int32)
        self.d  = np.empty(capacity, dtype=np.bool_)
        self.gp = np.empty(capacity, dtype=np.float32)   # γ^k
        self.pr = np.ones (capacity, dtype=np.float32)   # priorities

    def add(self, s, a, R, s2, d, gp, p=None):
        idx           = self.pos
        self.s [idx]  = s
        self.a [idx]  = a
        self.R [idx]  = R
        self.s2[idx]  = s2
        self.d [idx]  = d
        self.gp[idx]  = gp
        self.pr[idx]  = self.pr.max() if p is None else float(p)

        self.pos  = (self.pos + 1) % self.cap
        self.full = self.full or self.pos == 0

    def sample(self, batch: int, beta: float = 0.4):
        N = self.cap if self.full else self.pos
        if N < batch:
            return None, None, None

        pr_seg = self.pr[:N].astype(np.float32, copy=False)   # <- NEW (contiguous, right length)
        probs  = pr_seg ** self.alpha
        probs /= probs.sum()

        idxs  = np.random.choice(N, batch, replace=False, p=probs)
        w     = (N * probs[idxs]) ** (-beta)
        w    /= w.max()

        batch_dict = dict(
            s  = self.s [idxs],
            a  = self.a [idxs],
            R  = self.R [idxs],
            s2 = self.s2[idxs],
            d  = self.d [idxs],
            gp = self.gp[idxs]
        )
        return batch_dict, idxs, w.astype(np.float32)

    def update_priorities(self, idxs, td_err, eps=1e-6):
        self.pr[idxs] = np.abs(td_err).ravel() + eps

    def __len__(self): return self.cap if self.full else self.pos

# plain feed-forward Q-net
class QNet(nn.Module):
    def __init__(self, s_dim, a_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, a_dim)
        )
        self.apply(lambda m: nn.init.kaiming_uniform_(m.weight)
                   if isinstance(m, nn.Linear) else None)

    def forward(self, x): return self.net(x)

# Vectorised N-step DQN
class VectorNStepDQN:
    def __init__(self,
                 env_id           ='Sepsis/ICU-Sepsis-v2',
                 num_envs         = 8,
                 n_step           = 3,
                 gamma            = 0.99,
                 lr               = 5e-4,
                 buffer_size      = 200_000,
                 batch_size       = 128,
                 eps_start        = 1.0,
                 eps_end          = 0.05,
                 target_update    = 10_000,
                 learning_starts  = 10_000,
                 train_freq       = 4,
                 alpha=0.6, beta=0.4,
                 device='cpu'):

        self.num_envs = num_envs
        self.device   = torch.device(device)
        self.n        = n_step
        self.gamma    = gamma

        #  vector env
        def make(): return lambda: gym.make(env_id)
        self.envs = SyncVectorEnv([make() for _ in range(num_envs)])
        obs, _    = self.envs.reset()
        self.S = int(self.envs.single_observation_space.n)
        self.A = int(self.envs.single_action_space.n)

        #  nets & optim
        self.q    = QNet(self.S, self.A).to(self.device)
        self.tgt  = QNet(self.S, self.A).to(self.device)
        self.tgt.load_state_dict(self.q.state_dict())
        self.opt  = torch.optim.Adam(self.q.parameters(), lr=lr)

        #  replay
        self.replay = PrioritizedReplayNStep(buffer_size, alpha)
        self.batch_size      = batch_size
        self.beta            = beta

        #  hyper-param
        self.eps_start  = eps_start
        self.eps_end    = eps_end
        self.eps        = eps_start
        self.target_upd = target_update
        self.learn_st   = learning_starts
        self.train_freq = train_freq * num_envs   # global steps

        #  n-step helpers
        self.nbufs: List[NStepBuffer] = [NStepBuffer(n_step, gamma)
                                         for _ in range(num_envs)]

        #  trackers
        self.obs          = obs
        self.global_step  = 0
        self.learn_step   = 0
        self.global_epi   = 0
        self.max_episodes = None  # set by caller

        self.cur_ret = np.zeros(num_envs, dtype=np.float32)
        self.cur_len = np.zeros(num_envs, dtype=np.int32)
        self.ret_buf, self.len_buf = [], []

    def _decay_eps(self):
        frac   = min(1.0, self.global_epi / (0.25 * self.max_episodes))
        self.eps = self.eps_start + frac * (self.eps_end - self.eps_start)

    def select_actions(self, obs):
        if random.random() < self.eps:
            return np.random.randint(self.A, size=self.num_envs)
        with torch.no_grad():
            oh = one_hot(torch.as_tensor(obs, dtype=torch.long,
                                         device=self.device),
                         self.S, self.device)
            return torch.argmax(self.q(oh), 1).cpu().numpy()

    def step_envs(self):
        acts = self.select_actions(self.obs)
        nxt, rew, term, trunc, infos = self.envs.step(acts)
        done = term | trunc

        # update per-episode stats
        self.cur_ret += rew
        self.cur_len += 1
        fin = np.where(done)[0]
        if fin.size:
            self.ret_buf.extend(self.cur_ret[fin])
            self.len_buf.extend(self.cur_len[fin])
            self.cur_ret[fin] = 0.0
            self.cur_len[fin] = 0
            self.global_epi  += fin.size

        # push transitions into n-step buffers
        for i in range(self.num_envs):
            for tr in self.nbufs[i].step(self.obs[i], acts[i],
                                         rew[i], done[i]):
                self.replay.add(*tr)

        self.obs = nxt
        self.global_step += self.num_envs
        self._decay_eps()

        # maybe learn
        if (self.global_step > self.learn_st and
            self.global_step % self.train_freq == 0):
            self.learn()

        return rew.mean(), done.mean()

    def learn(self):
        batch, idxs, w = self.replay.sample(self.batch_size, self.beta)
        if batch is None:                         # buffer not ready yet
            return

        s  = torch.as_tensor(batch['s'],  dtype=torch.long,   device=self.device)
        a  = torch.as_tensor(batch['a'],  dtype=torch.long,   device=self.device).unsqueeze(1)
        R  = torch.as_tensor(batch['R'],  dtype=torch.float32,device=self.device).unsqueeze(1)
        s2 = torch.as_tensor(batch['s2'], dtype=torch.long,   device=self.device)
        d  = torch.as_tensor(batch['d'],  dtype=torch.float32,device=self.device).unsqueeze(1)
        gp = torch.as_tensor(batch['gp'], dtype=torch.float32,device=self.device).unsqueeze(1)
        w  = torch.as_tensor(w,          dtype=torch.float32,device=self.device).unsqueeze(1)

        q   = self.q (one_hot(s,  self.S, self.device)).gather(1, a)
        with torch.no_grad():
            qn  = self.tgt(one_hot(s2, self.S, self.device)).max(1, keepdim=True)[0]
            tgt = R + gp * qn * (1.0 - d)        # gp = γ^k

        td   = tgt - q
        loss = (w * td.pow(2)).mean()

        self.opt.zero_grad()
        loss.backward()
        self.opt.step()

        self.replay.update_priorities(idxs, td.detach().cpu().numpy())
        self.learn_step += 1
        if self.learn_step % self.target_upd == 0:
            self.tgt.load_state_dict(self.q.state_dict())

In [ ]:
N = 3
agent = VectorNStepDQN(num_envs=8, n_step=N, gamma=0.99,
                       lr=5e-4)
agent.max_episodes = 300_000

pbar = tqdm(total=agent.max_episodes, desc=f'Vector-{N}-Step-DQN')
while agent.global_epi < agent.max_episodes:
    r_mean, d_mean = agent.step_envs()
    pbar.update(d_mean * agent.num_envs)   # episodes finished since last step
pbar.close()

# episode-level curves
returns = np.array(agent.ret_buf)
lengths = np.array(agent.len_buf)


In [ ]:
results = np.array(agent.ret_buf)
r_mean = results.reshape(-1, 1).mean(axis=1)

def smooth(x, w=10000):
    return np.convolve(x, np.ones(w)/w, mode='same')

sr = smooth(np.array(results))

# 1. Survival Rate
plt.figure()
plt.plot(sr)
plt.title('Survival Rate (Single Seed)')
plt.xlabel('Episode')
plt.ylabel('Return')
plt.ylim(0.74, 0.86)
plt.grid(True)
plt.show()

In [ ]:
base_dir = "/content/drive/MyDrive/COMP_579_Final_Project/Multi-Step DQN"
os.makedirs(base_dir, exist_ok=True)

# sweep ranges
seeds         = range(10)
learning_rates= [1e-3, 5e-4]
n_steps       = [3, 5, 7]              # horizon sweep
gamma         = 0.99                   # fixed

episodes_total     = 300_000           # completed episodes
checkpoint_every   = 100_000           # completed episodes
batch_size         = 128               # keep fixed
num_envs           = 8

for n in n_steps:
    for lr in learning_rates:
        tag      = f"NStep{n}_lr{lr}_bs{batch_size}_gamma{gamma}"
        ckpt_dir = os.path.join(base_dir, tag)

        for seed in seeds:
            # reproducibility
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

            # metric files
            f_ret  = os.path.join(ckpt_dir, f"{tag}_seed{seed}_returns.npy")
            f_len  = os.path.join(ckpt_dir, f"{tag}_seed{seed}_lengths.npy")
            f_inad = os.path.join(ckpt_dir, f"{tag}_seed{seed}_inad.npy")

            # resume metrics if present
            returns = list(np.load(f_ret))  if os.path.exists(f_ret)  else []
            lengths = list(np.load(f_len))  if os.path.exists(f_len)  else []
            inads   = list(np.load(f_inad)) if os.path.exists(f_inad) else []

            # find latest checkpoint
            last_ckpt = 0
            if os.path.isdir(ckpt_dir):
                pat = re.compile(
                    fr"{re.escape(tag)}_seed{seed}_epi(\d+)_q\.pth")
                for fn in os.listdir(ckpt_dir):
                    m = pat.match(fn)
                    if m:
                        last_ckpt = max(last_ckpt, int(m.group(1)))

            # instantiate agent
            agent = VectorNStepDQN(num_envs        = num_envs,
                                   n_step          = n,
                                   gamma           = gamma,
                                   lr              = lr,
                                   batch_size      = batch_size,
                                   buffer_size     = 200_000,
                                   device          = "cpu")
            agent.max_episodes = episodes_total

            # restore nets if checkpoint exists
            if last_ckpt:
                agent.q.load_state_dict(
                    torch.load(os.path.join(
                        ckpt_dir, f"{tag}_seed{seed}_epi{last_ckpt}_q.pth"),
                        map_location=agent.device))
                agent.tgt.load_state_dict(
                    torch.load(os.path.join(
                        ckpt_dir, f"{tag}_seed{seed}_epi{last_ckpt}_target.pth"),
                        map_location=agent.device))
                agent.global_epi = last_ckpt   # keep ε schedule

            pbar = tqdm(total=episodes_total-agent.global_epi,
                        desc=f"{tag} | seed{seed}", unit='ep')

            while agent.global_epi < episodes_total:
                r_mean, d_mean = agent.step_envs()

                # flush finished episodes into lists
                while len(returns) < len(agent.ret_buf):
                    idx = len(returns)
                    returns.append(float(agent.ret_buf[idx]))
                    lengths.append(int  (agent.len_buf[idx]))
                    inads.append(0.0)                    # not tracked
                    pbar.update(1)

                # checkpoint
                if (agent.global_epi and
                    agent.global_epi % checkpoint_every == 0):
                    os.makedirs(ckpt_dir, exist_ok=True)
                    torch.save(agent.q.state_dict(),
                               os.path.join(ckpt_dir,
                               f"{tag}_seed{seed}_epi{agent.global_epi}_q.pth"))
                    torch.save(agent.tgt.state_dict(),
                               os.path.join(ckpt_dir,
                               f"{tag}_seed{seed}_epi{agent.global_epi}_target.pth"))
                    np.save(f_ret,  np.array(returns))
                    np.save(f_len,  np.array(lengths))
                    np.save(f_inad, np.array(inads))

            pbar.close()

            # final save
            os.makedirs(ckpt_dir, exist_ok=True)
            torch.save(agent.q.state_dict(),
                       os.path.join(ckpt_dir,
                       f"{tag}_seed{seed}_final_q.pth"))
            torch.save(agent.tgt.state_dict(),
                       os.path.join(ckpt_dir,
                       f"{tag}_seed{seed}_final_target.pth"))
            np.save(f_ret,  np.array(returns))
            np.save(f_len,  np.array(lengths))
            np.save(f_inad, np.array(inads))

In [ ]:
base_dir       = Path("/content/drive/MyDrive/COMP_579_Final_Project/Multi-Step DQN")
seeds          = range(10)
learning_rates = [1e-3, 5e-4]
n_steps        = [3, 5, 7]          # horizons that were trained
gamma          = 0.99
batch_size     = 128
metric_kind    = "returns"          # "returns" or "lengths"
smooth_window  = 5_000              # change to 10_000 for lengths if desired
ylim_dict      = {"returns":(0.74,0.88), "lengths":(9,14)}  # y-axis limits

def smooth(x, w):
    if len(x) < w:
        return x                                       # nothing to do
    return np.convolve(x, np.ones(w)/w, mode="same")   # centred moving-avg

# collect all tags (= unique experiment folders)
tags = [f"NStep{n}_lr{lr}_bs{batch_size}_gamma{gamma}"
        for n in n_steps for lr in learning_rates]

plt.figure(figsize=(11, 6))
cmap = plt.get_cmap("tab10")       # safe on all Matplotlib versions

for i, tag in enumerate(tags):
    runs   = []                    # holds 1D metric arrays for each seed
    for seed in seeds:
        f = base_dir / tag / f"{tag}_seed{seed}_{metric_kind}.npy"
        if f.exists():
            arr = np.load(f)
            runs.append(arr.astype(float))             # cast → float for NaN padding

    if not runs:
        print(f"[warn] No '{metric_kind}' files found for {tag}")
        continue

    # pad all runs to the same length with NaNs so stack → nanmean works
    max_len = max(len(r) for r in runs)
    runs = [np.pad(r, (0, max_len - len(r)), constant_values=np.nan) for r in runs]
    runs = np.vstack(runs)                             # shape (n_seeds, max_len)

    # compute mean / SEM ignoring NaNs
    mean   = np.nanmean(runs, 0)
    sem    = np.nanstd (runs, 0) / np.sqrt(np.sum(~np.isnan(runs), 0))

    # smooth only the mean & SEM
    mean_s = smooth(mean, smooth_window)
    sem_s  = smooth(sem , smooth_window)

    x = np.arange(len(mean_s))
    colour = cmap(i % cmap.N)

    plt.plot(x, mean_s, color=colour, label=tag, lw=1.4)
    plt.fill_between(x, mean_s-sem_s, mean_s+sem_s,
                     color=colour, alpha=0.15)

title = ("Survival-rate" if metric_kind=="returns" else "Episode-length")
plt.title(f"{title} curves • Multi-Step DQN hyper-parameter sweep")
plt.xlabel("Episode")
plt.ylabel("Return" if metric_kind=="returns" else "Episode length")
plt.grid(True, alpha=.3)
if metric_kind in ylim_dict: plt.ylim(*ylim_dict[metric_kind])
plt.legend(bbox_to_anchor=(1.02,1), loc="upper left", fontsize="small")
plt.tight_layout()
plt.show()

In [ ]:
base_dir       = Path("/content/drive/MyDrive/COMP_579_Final_Project/Multi-Step DQN")
seeds          = range(10)
learning_rates = [1e-3, 5e-4]
n_steps        = [3, 5, 7]          # horizons that were trained
gamma          = 0.99
batch_size     = 128
metric_kind    = "lengths"          # "returns" or "lengths"
smooth_window  = 5_000              # change to 10_000 for lengths if desired
ylim_dict      = {"returns":(0.74,0.88), "lengths":(9,14)}  # y-axis limits

def smooth(x, w):
    if len(x) < w:
        return x                                       # nothing to do
    return np.convolve(x, np.ones(w)/w, mode="same")   # centred moving-avg

# collect all tags (= unique experiment folders)
tags = [f"NStep{n}_lr{lr}_bs{batch_size}_gamma{gamma}"
        for n in n_steps for lr in learning_rates]

plt.figure(figsize=(11, 6))
cmap = plt.get_cmap("tab10")       # safe on all Matplotlib versions

for i, tag in enumerate(tags):
    runs   = []                    # holds 1D metric arrays for each seed
    for seed in seeds:
        f = base_dir / tag / f"{tag}_seed{seed}_{metric_kind}.npy"
        if f.exists():
            arr = np.load(f)
            runs.append(arr.astype(float))             # cast → float for NaN padding

    if not runs:
        print(f"[warn] No '{metric_kind}' files found for {tag}")
        continue

    # pad all runs to the same length with NaNs so stack → nanmean works
    max_len = max(len(r) for r in runs)
    runs = [np.pad(r, (0, max_len - len(r)), constant_values=np.nan) for r in runs]
    runs = np.vstack(runs)                             # shape (n_seeds, max_len)

    # compute mean / SEM ignoring NaNs
    mean   = np.nanmean(runs, 0)
    sem    = np.nanstd (runs, 0) / np.sqrt(np.sum(~np.isnan(runs), 0))

    # smooth only the mean & SEM
    mean_s = smooth(mean, smooth_window)
    sem_s  = smooth(sem , smooth_window)

    x = np.arange(len(mean_s))
    colour = cmap(i % cmap.N)

    plt.plot(x, mean_s, color=colour, label=tag, lw=1.4)
    plt.fill_between(x, mean_s-sem_s, mean_s+sem_s,
                     color=colour, alpha=0.15)

# final plot cosmetics
title = ("Survival-rate" if metric_kind=="returns" else "Episode-length")
plt.title(f"{title} curves • Multi-Step DQN hyper-parameter sweep")
plt.xlabel("Episode")
plt.ylabel("Return" if metric_kind=="returns" else "Episode length")
plt.grid(True, alpha=.3)
if metric_kind in ylim_dict: plt.ylim(*ylim_dict[metric_kind])
plt.legend(bbox_to_anchor=(1.02,1), loc="upper left", fontsize="small")
plt.tight_layout()
plt.show()